# Exiting muons: MCS vs MCS+CSDA

This notebook studies **partially contained (exiting) muons** only (no fully-contained stopping tracks).
We compare MCS-only and a combined MCS+CSDA fit across **various true energies** and **various visible track fractions**.

## Mathematical idea

For segment-angle observations $\{\theta_i\}$ and initial kinetic energy $T_0$:

$$\hat T_0^{\mathrm{MCS}} = \arg\min_{T_0}\;\mathcal{L}_{\mathrm{MCS}}(T_0;\{\theta_i\}).$$

For an exiting track, only visible range $R_{\mathrm{vis}}$ is known. We map it to a CSDA proxy energy
$T_{\mathrm{CSDA}} = f_{\mathrm{CSDA}}(R_{\mathrm{vis}})$ and build a joint objective:

$$\mathcal{L}_{\mathrm{joint}}(T_0)=\mathcal{L}_{\mathrm{MCS}}(T_0)+\frac{w}{2}\left(\frac{T_0-T_{\mathrm{CSDA}}}{\sigma_{\mathrm{CSDA}}}\right)^2,$$

with $\sigma_{\mathrm{CSDA}}=\max(1, f\,T_{\mathrm{CSDA}})$.

So we are effectively doing a MAP estimate using MCS likelihood + CSDA prior, then evaluating resolution on exiting tracks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from spine.utils.energy_loss import csda_range_lar, csda_table_spline, step_energy_loss_lar
from spine.utils.globals import MUON_MASS, MUON_PID
from spine.utils.mcs import highland, mcs_fit

rng = np.random.default_rng(20260409)
plt.style.use('seaborn-v0_8-whitegrid')
csda_spline = csda_table_spline(MUON_PID)

## Toy setup

We sample:
- true muon KE: 150 to 1200 MeV
- visible fraction (contained length / true total range): 0.2 to 0.9

For each toy, only the visible segment contributes to scattering and CSDA proxy.
We compare:
- **MCS-only**
- **MCS+CSDA** (Gaussian CSDA term).

In [ ]:
def simulate_exiting_muon(T0_true, frac_contained, dx=5.0):
    # Full range for a contained equivalent track
    R_true = csda_range_lar(T0_true, MUON_MASS)

    # Visible (contained) range for exiting track
    R_vis = max(dx, frac_contained * R_true)
    T_vis = float(csda_spline(R_vis))

    # Build scattering sequence only along visible range
    n_steps = max(4, int(R_vis / dx))
    ke = step_energy_loss_lar(T0_true, MUON_MASS, dx, num_steps=n_steps)
    if len(ke) < n_steps + 1:
        return None

    p = np.sqrt(ke**2 + 2.0 * MUON_MASS * ke)
    p_steps = np.sqrt(p[:-1] * p[1:])

    # Match likelihood model (Highland + angular resolution)
    theta0 = highland(p_steps, MUON_MASS, dx)
    res = 0.25 / dx**1.25
    theta_obs = rng.rayleigh(np.sqrt(theta0**2 + res**2))

    # MCS-only
    ke_mcs = mcs_fit(
        theta_obs,
        MUON_MASS,
        dx,
        res_a=0.25,
        res_b=1.25,
        lower_bound=10.0,
        upper_bound=3000.0,
    )

    # Combined MCS + CSDA Gaussian prior
    ke_comb = mcs_fit(
        theta_obs,
        MUON_MASS,
        dx,
        res_a=0.25,
        res_b=1.25,
        csda_ke=T_vis,
        csda_ke_frac=0.40,
        csda_weight=5.0,
        csda_mode='gaussian',
        lower_bound=10.0,
        upper_bound=3000.0,
    )

    return {
        'T0_true': T0_true,
        'frac_contained': frac_contained,
        'R_true': R_true,
        'R_vis': R_vis,
        'T_vis_csda': T_vis,
        'mcs_ke': ke_mcs,
        'comb_ke': ke_comb,
        'n_steps': n_steps,
    }

In [ ]:
records = []
for _ in range(1400):
    T0 = rng.uniform(150.0, 1200.0)
    frac = rng.uniform(0.2, 0.9)
    rec = simulate_exiting_muon(T0, frac, dx=5.0)
    if rec is not None:
        records.append(rec)

df = pd.DataFrame(records)
df['err_mcs'] = df['mcs_ke'] - df['T0_true']
df['err_comb'] = df['comb_ke'] - df['T0_true']
df['abs_err_mcs'] = np.abs(df['err_mcs'])
df['abs_err_comb'] = np.abs(df['err_comb'])

summary = pd.DataFrame({
    'metric': ['MAE', 'RMSE', 'Median |err|'],
    'MCS only': [
        df['abs_err_mcs'].mean(),
        np.sqrt(np.mean(df['err_mcs']**2)),
        np.median(df['abs_err_mcs']),
    ],
    'MCS + CSDA': [
        df['abs_err_comb'].mean(),
        np.sqrt(np.mean(df['err_comb']**2)),
        np.median(df['abs_err_comb']),
    ],
})
summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (1) Residual histogram
axes[0, 0].hist(df['err_mcs'], bins=60, alpha=0.55, label='MCS only')
axes[0, 0].hist(df['err_comb'], bins=60, alpha=0.55, label='MCS+CSDA')
axes[0, 0].axvline(0.0, color='k', lw=1)
axes[0, 0].set_title('Residuals: fitted KE - true KE')
axes[0, 0].set_xlabel('MeV')
axes[0, 0].legend()

# (2) Absolute error vs true energy
axes[0, 1].scatter(df['T0_true'], df['abs_err_mcs'], s=8, alpha=0.35, label='MCS only')
axes[0, 1].scatter(df['T0_true'], df['abs_err_comb'], s=8, alpha=0.35, label='MCS+CSDA')
axes[0, 1].set_yscale('log')
axes[0, 1].set_xlabel('True KE [MeV]')
axes[0, 1].set_ylabel('|error| [MeV]')
axes[0, 1].set_title('Absolute error vs true energy')
axes[0, 1].legend()

# (3) Error by contained fraction bin
bins = np.array([0.2, 0.35, 0.5, 0.65, 0.8, 0.9])
idx = np.digitize(df['frac_contained'], bins)
centers = 0.5 * (bins[:-1] + bins[1:])
mcs_bin = []
comb_bin = []
for b in range(1, len(bins)):
    sel = idx == b
    mcs_bin.append(np.median(df.loc[sel, 'abs_err_mcs']))
    comb_bin.append(np.median(df.loc[sel, 'abs_err_comb']))
axes[1, 0].plot(centers, mcs_bin, 'o-', label='MCS only')
axes[1, 0].plot(centers, comb_bin, 'o-', label='MCS+CSDA')
axes[1, 0].set_xlabel('Contained fraction (visible/total range)')
axes[1, 0].set_ylabel('Median |error| [MeV]')
axes[1, 0].set_title('Resolution vs containment fraction')
axes[1, 0].legend()

# (4) Truth vs fit
axes[1, 1].scatter(df['T0_true'], df['mcs_ke'], s=8, alpha=0.35, label='MCS only')
axes[1, 1].scatter(df['T0_true'], df['comb_ke'], s=8, alpha=0.35, label='MCS+CSDA')
lims = [150, 1200]
axes[1, 1].plot(lims, lims, 'k--', lw=1)
axes[1, 1].set_xlim(lims)
axes[1, 1].set_ylim(lims)
axes[1, 1].set_xlabel('True KE [MeV]')
axes[1, 1].set_ylabel('Fitted KE [MeV]')
axes[1, 1].set_title('Truth vs reconstruction')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## Notes

- This study intentionally excludes fully-contained tracks.
- The CSDA term is computed from **visible** range only, so it is a proxy for exiting muons; its pull is controlled by `csda_ke_frac` and `csda_weight`.
- The resolution-vs-contained-fraction plot lets you inspect where combination helps most.